### Построение базовой модели

In [1]:
import numpy as np
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

np.random.seed = 42

In [3]:
with open('./data/train_data.json', 'r', encoding='utf-8') as file:
    train_data = json.load(file)

X_train = train_data['X']
y_train = train_data['y']

print(len(X_train), len(y_train))

with open('./data/test_data.json', 'r', encoding='utf-8') as file:
    test_data = json.load(file)

X_test = test_data['X']
y_test = test_data['y']

print(len(X_test), len(y_test))

25000 25000
25000 25000


In [4]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',    # нормализация символов
    ngram_range=(1, 2),         # униграммы + биграммы
    min_df=2,                   # отсечение очень редкие токены
    max_df=0.95,                # отсекчение слишком частые токены
    max_features=50000,         # ограничение размерности
    sublinear_tf=True
)

pipe = Pipeline([
    ('tfidf', tfidf),
    ('clf', LinearSVC())
])

param_grid = [

    # LinearSVC
    {
        'clf': [LinearSVC()],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    },

    # Logistic Regression
    {
        'clf': [LogisticRegression(max_iter=2000, solver='liblinear')],
        'clf__C': [0.1, 0.5, 1, 2],
        'tfidf__ngram_range': [(1,1), (1,2)],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.9, 0.95],
        'tfidf__max_features': [30000, 50000, 70000, 100000] 
    }
]

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print('\nЛучшие параметры:')
print(grid.best_params_)

print('\nЛучший CV score:')
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print('\nTest Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

Fitting 3 folds for each of 384 candidates, totalling 1152 fits

Лучшие параметры:
{'clf': LinearSVC(), 'clf__C': 0.1, 'tfidf__max_df': 0.95, 'tfidf__max_features': 30000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}

Лучший CV score:
0.8822000396214552

Test Accuracy: 0.90188

Classification Report:
              precision    recall  f1-score   support

           0     0.9066    0.8961    0.9013     12500
           1     0.8973    0.9077    0.9024     12500

    accuracy                         0.9019     25000
   macro avg     0.9019    0.9019    0.9019     25000
weighted avg     0.9019    0.9019    0.9019     25000



#### После проведения тестов, лучшей базовой моделью оказалась LinearSVC с параметрами: 

- C = 0.1

#### и TfIdf векторизатором с параметрами:

- ngram_range = (1, 2)

- max_df = 0.9

- min_df = 1

- max_features = 30000

### Построение CNN модели

In [5]:
import re

from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

Функции кодирования слов и создания словаря

In [6]:
_token_re = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    # простая англ. токенизация: слова и апострофы
    return _token_re.findall(text.lower())

PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

def build_vocab(texts, min_freq=2, max_size=50000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    # самые частые слова в словарь, но частота их должна быть больше min_freq
    most_common = [w for w, c in counter.most_common() if c >= min_freq]
    # обрезаем словарь частых слов по максимальной длине словаря max_size
    if max_size is not None:
        most_common = most_common[:max_size]
    
    # создаем словарь индексов
    stoi = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    # присваиваем каждому слову уникальный номер
    for w in most_common:
        if w not in stoi:
            stoi[w] = len(stoi)
    return stoi

def encode(text, stoi, max_len):
    tokens = tokenize(text)
    # перевод слов в числа с помощью словаря stoi
    ids = [stoi.get(w, stoi[UNK_TOKEN]) for w in tokens]
    # обрезаем если отзыв слишком длинный
    ids = ids[:max_len]
    
    # если отзыв короткий, добавляем нули
    if len(ids) < max_len:
        ids += [stoi[PAD_TOKEN]] * (max_len - len(ids))

Класс датасета и класс CNN модели

In [17]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        print(self.stoi)
        X = torch.tensor(encode(self.texts[index], self.stoi, self.max_len), dtype=torch.long)
        y = torch.tensor(self.labels[index], dtype=torch.long)
        return X, y
    
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        num_classes: int = 2,
        kernel_sizes=(3, 4, 5),
        num_filters: int = 128,
        dropout: float = 0.5,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=ks)
            for ks in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)
        self.fc =nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.transpose(1, 2)

        pooled = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            p = torch.max(h, dim=2).values
            pooled.append(p)

        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        logits = self.fc(out)

        return logits

Функции обучения и валидации модели

In [8]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    crossentropy = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = crossentropy(logits, y)

        loss_sum += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return loss_sum / total, correct / total

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    crossentropy = nn.CrossEntropyLoss()

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = crossentropy(logits, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return loss_sum / total, correct / total

Создание словаря и модели, подбор гиперпараметров, обучение и оценка

In [18]:
max_len = 400
min_freq = 2
max_size = 50000
batch_size = 64
embed_dim = 128
num_filters = 128
kernel_sizes = (3, 4, 5)
dropout = 0.5
lr = 1e-3
epochs = 10

stoi = build_vocab(X_train, min_freq, max_size)
vocab_size = len(stoi)

train_ds = TextDataset(X_train, y_train, stoi, max_len)
test_ds = TextDataset(X_test, y_test, stoi, max_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

modelCNN = TextCNN(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_classes=2,
    kernel_sizes=kernel_sizes,
    num_filters=num_filters,
    dropout=dropout,
    pad_idx=stoi[PAD_TOKEN],
).to(device)

optimizer_AdamW = torch.optim.AdamW(modelCNN.parameters(), lr=lr)

for epoch in range(1, epochs + 1):
    train_loss, train_accuracy = train_one_epoch(modelCNN, train_loader, optimizer_AdamW, device)
    test_loss, test_accuracy = evaluate(modelCNN, test_loader, device)
    print(f'Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_accuracy:.4f} | test loss {test_loss:.4f} acc {test_accuracy:.4f}')

{'<pad>': 0, '<unk>': 1, 'the': 2, 'and': 3, 'a': 4, 'of': 5, 'to': 6, 'is': 7, 'in': 8, 'it': 9, 'i': 10, 'this': 11, 'that': 12, 'was': 13, 'as': 14, 'for': 15, 'with': 16, 'movie': 17, 'but': 18, 'film': 19, 'on': 20, 'not': 21, 'you': 22, 'are': 23, 'his': 24, 'have': 25, 'he': 26, 'be': 27, 'one': 28, 'all': 29, 'at': 30, 'by': 31, 'an': 32, 'they': 33, 'who': 34, 'so': 35, 'from': 36, 'like': 37, 'her': 38, 'or': 39, 'just': 40, 'about': 41, "it's": 42, 'out': 43, 'if': 44, 'has': 45, 'some': 46, 'there': 47, 'what': 48, 'good': 49, 'more': 50, 'when': 51, 'very': 52, 'up': 53, 'no': 54, 'time': 55, 'she': 56, 'even': 57, 'my': 58, 'would': 59, 'which': 60, 'only': 61, 'story': 62, 'really': 63, 'see': 64, 'their': 65, 'had': 66, 'can': 67, 'were': 68, 'me': 69, 'well': 70, 'than': 71, 'we': 72, 'much': 73, 'been': 74, 'bad': 75, 'get': 76, 'will': 77, 'do': 78, 'also': 79, 'into': 80, 'people': 81, 'other': 82, 'first': 83, 'because': 84, 'great': 85, 'him': 86, 'how': 87, 'most

TypeError: 'NoneType' object cannot be interpreted as an integer